# Set-Up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import RadarPointCloud
from nuscenes.utils.geometry_utils import points_in_box

In [ ]:
%matplotlib inline
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version = 'v1.0-mini', dataroot = DATAROOT, verbose = True)

In [ ]:
DATAROOT = r"data/sets/nuscenes"
VERSION = "v1.0-mini"
SCENE_INDEX = 0
RADAR_CHAN = "RADAR_FRONT"
REF_CHAN = "LIDAR_TOP"
REF_SENSOR = "LIDAR_TOP"
SENSORS_TO_EVAL = ["CAM_FRONT", "RADAR_FRONT"]
THRESHOLD_MS = 50
MIN_RADAR_HITS = 1

scene_token = nusc.scene[0]["token"]

In [ ]:
# Helper Function

def get_scene_samples(nusc, scene_index = 0):
    """
    Returns all samples from a selected nuScenes scene in temporal order.
    """
    scene = nusc.scene[scene_index]
    token = scene["first_sample_token"]

    samples = []

    while token:
        sample = nusc.get("sample", token)
        samples.append(sample)
        token = sample["next"]

    return samples

# 5.1 Fusion Methods

In [ ]:
# Appendix Code 1: Baseline LiDAR-Radar Object-Level Fusion

def fuse_frame(nusc, sample, radar_chan = "RADAR_FRONT", ref_chan = "LIDAR_TOP", min_hits = 1):
    """
    Performs baseline object-level LiDAR-radar fusion for one frame.

    LiDAR bounding boxes are used as spatial object references. Radar points are
    associated with an object if they fall inside the corresponding LiDAR box.
    The fused object position is taken from the LiDAR box centre, while velocity
    is estimated from the associated radar measurements.
    """
    lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])

    # Load one radar sweep transformed into the LiDAR reference frame.
    radar_pc, _ = RadarPointCloud.from_file_multisweep(
        nusc,
        sample,
        chan = radar_chan,
        ref_chan = ref_chan,
        nsweeps = 1
    )

    radar_xyz = radar_pc.points[:3, :]
    radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))

    # Load LiDAR-frame 3D object boxes.
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    fused_objects = []

    for box_index, box in enumerate(boxes):
        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        # Median velocity is used as a simple robust estimate.
        velocity_xy = np.median(radar_vel_xy[:, inside_box], axis = 1)

        fused_objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": box.center[:2].copy(),
            "velocity_xy": velocity_xy.copy(),
            "radar_hits": hit_count
        })

    return fused_objects


def run_scene_baseline(nusc, scene_index = 0, radar_chan = "RADAR_FRONT",
                       ref_chan = "LIDAR_TOP", min_hits = 1):
    """
    Applies the baseline fusion method to every frame in a selected nuScenes scene.
    """
    scene = nusc.scene[scene_index]
    sample_token = scene["first_sample_token"]

    all_frames = []
    frame_index = 0

    while sample_token:
        sample = nusc.get("sample", sample_token)

        fused_objects = fuse_frame(
            nusc,
            sample,
            radar_chan = radar_chan,
            ref_chan = ref_chan,
            min_hits = min_hits
        )

        all_frames.append({
            "frame_index": frame_index,
            "sample_token": sample_token,
            "objects": fused_objects
        })

        sample_token = sample["next"]
        frame_index += 1

    return all_frames


def summarise_fusion_results(all_frames):
    """
    Computes summary metrics used to describe the baseline fusion output.
    """
    hits = []
    speeds = []

    for frame in all_frames:
        for obj in frame["objects"]:
            hits.append(obj["radar_hits"])

            vx, vy = obj["velocity_xy"]
            speeds.append(np.sqrt(vx**2 + vy**2))

    return {
        "total_frames": len(all_frames),
        "total_fused_objects": sum(len(frame["objects"]) for frame in all_frames),
        "average_radar_hits": float(np.mean(hits)) if hits else 0.0,
        "median_radar_hits": float(np.median(hits)) if hits else 0.0,
        "average_speed_magnitude": float(np.mean(speeds)) if speeds else 0.0,
        "median_speed_magnitude": float(np.median(speeds)) if speeds else 0.0
    }

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.498 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.
Processing scene: scene-0061

Frame 0 | token=ca9a28 | fused_objects=9
Obj 9 | class=movable_object.barrier | pos=[ 7.85689816 25.55669921] | vel=[-0.15958051  0.05388169] | hits=2
Obj 16 | class=vehicle.car | pos=[ 5.97927366 35.00872533] | vel=[-0.01604966  0.00264043] | hits=1
Obj 18 | class=vehicle.truck | pos=[-4.4986433  15.25332251] | vel=[-0.16419582 -0.04946925] | hits=7
Obj 21 | class=movable_object.barrier | pos=[ 9.53171813 42.28333322] | vel=[-0.16745839  0.0383587 ] | hits=1
Obj 25 | class=movable_object.barrier | pos=[ 7.09060535 15.51871913] | vel=[-0.16145445  0.08756081] | hits=2
Obj 59 | class=movable_object.pushable_pullable | pos=[-2.80841235 16.74

# 5.2 Temporal Alignment

In [ ]:
# Appendix Code 2: Temporal Alignment and Continuity-Based Offset Correction


# Clean and corrupted timestamp pairing

def build_scene_timestamp_table(nusc, scene_token):
    """
    Builds a per-frame timestamp table for all sensors in a scene.
    Timestamps are stored in microseconds.
    """
    scene = nusc.get("scene", scene_token)
    sample_token = scene["first_sample_token"]

    rows = []

    while sample_token:
        sample = nusc.get("sample", sample_token)

        row = {
            "sample_token": sample_token,
            "sample_timestamp": int(sample["timestamp"])
        }

        for channel, sd_token in sample["data"].items():
            sample_data = nusc.get("sample_data", sd_token)
            row[channel] = int(sample_data["timestamp"])

        rows.append(row)
        sample_token = sample["next"]

    return pd.DataFrame(rows).sort_values("sample_timestamp").reset_index(drop = True)


def match_nearest_timestamps(ref_ts_us, target_ts_us, threshold_ms = 50.0):
    """
    Matches each reference timestamp to the nearest target timestamp.
    Returns signed and absolute timestamp error in milliseconds.
    """
    ref = np.asarray(ref_ts_us, dtype = "float64")
    target = np.asarray(target_ts_us, dtype = "float64")
    target = target[np.isfinite(target)]

    result = pd.DataFrame({
        "matched": np.zeros(len(ref), dtype = bool),
        "dt_ms_signed": np.full(len(ref), np.nan),
        "dt_ms_abs": np.full(len(ref), np.nan)
    })

    if len(target) == 0:
        return result

    target.sort()
    threshold_us = threshold_ms * 1000.0

    for i, t_ref in enumerate(ref):
        if not np.isfinite(t_ref):
            continue

        j = np.searchsorted(target, t_ref)

        candidates = []
        if j > 0:
            candidates.append(target[j - 1])
        if j < len(target):
            candidates.append(target[j])

        best = min(candidates, key = lambda t: abs(t - t_ref))
        dt_us = best - t_ref

        if abs(dt_us) <= threshold_us:
            result.loc[i, "matched"] = True
            result.loc[i, "dt_ms_signed"] = dt_us / 1000.0
            result.loc[i, "dt_ms_abs"] = abs(dt_us) / 1000.0

    return result


def corrupt_timestamps(df, sensors, jitter_ms = 20.0, drop_prob = 0.05, seed = 42):
    """
    Creates a corrupted timestamp table using jitter and random frame drop.
    This was used to compare clean and corrupted timestamp pairing.
    """
    rng = np.random.default_rng(seed)
    out = df.copy()

    for sensor in sensors:
        timestamps = out[sensor].to_numpy(dtype = "float64")

        valid = np.isfinite(timestamps)

        if drop_prob > 0:
            drop_mask = rng.random(len(timestamps)) < drop_prob
            timestamps[valid & drop_mask] = np.nan

        valid = np.isfinite(timestamps)

        if jitter_ms > 0:
            timestamps[valid] += rng.normal(
                0.0,
                jitter_ms * 1000.0,
                size = np.sum(valid)
            )

        out[sensor] = timestamps

    return out


def evaluate_timestamp_pairing(df, ref_sensor, target_sensors, threshold_ms = 50.0):
    """
    Summarises timestamp matching quality for each target sensor.
    """
    rows = []
    matches = {}

    ref_ts = df[ref_sensor].to_numpy(dtype = "float64")

    for sensor in target_sensors:
        target_ts = df[sensor].to_numpy(dtype = "float64")
        matched = match_nearest_timestamps(ref_ts, target_ts, threshold_ms)

        matches[sensor] = matched

        rows.append({
            "sensor": sensor,
            "match_rate_pct": float(matched["matched"].mean() * 100.0),
            "mean_abs_dt_ms": float(np.nanmean(matched["dt_ms_abs"])),
            "p95_abs_dt_ms": float(np.nanpercentile(matched["dt_ms_abs"], 95)),
            "max_abs_dt_ms": float(np.nanmax(matched["dt_ms_abs"]))
        })

    return pd.DataFrame(rows), matches



def summarise_alignment(df, col = "alignment_error_xy"):
    """
    Returns mean alignment error used for temporal comparison.
    """
    values = df[col].to_numpy()
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.mean(values))


def drift_offset_fn(a_ms_per_frame = 0.5, b_ms = 0.0, jitter_ms = 5.0, seed = 42):
    """
    Simulates timestamp drift:
        offset(i) = b + a*i + Gaussian noise
    """
    rng = np.random.default_rng(seed)

    def fn(i):
        drift = b_ms + a_ms_per_frame * i
        noise = rng.normal(0.0, jitter_ms)
        return float(drift + noise)

    return fn


def smooth_offsets(offsets, alpha = 0.25):
    """
    Exponential smoothing of frame-wise offset estimates.
    """
    offsets = np.asarray(offsets, dtype = float)
    smoothed = np.zeros_like(offsets)
    smoothed[0] = offsets[0]

    for i in range(1, len(offsets)):
        smoothed[i] = alpha * offsets[i] + (1 - alpha) * smoothed[i - 1]

    return smoothed


def estimate_offsets_with_continuity(raw_errors, candidate_offsets, lam = 0.02):
    """
    Greedy continuity-aware offset selection.

    Adds a penalty for large frame-to-frame offset jumps:
        cost = alignment_error + λ * |offset_t - offset_(t-1)|
    """
    candidate_offsets = np.asarray(candidate_offsets, dtype = float)
    n_frames = len(next(iter(raw_errors.values())))

    chosen = np.zeros(n_frames)

    # Frame 0: choose minimum raw error
    first_errors = np.array([raw_errors[o][0] for o in candidate_offsets])
    chosen[0] = candidate_offsets[np.argmin(first_errors)]

    for i in range(1, n_frames):
        previous = chosen[i - 1]

        costs = [
            raw_errors[o][i] + lam * abs(o - previous)
            for o in candidate_offsets
        ]

        chosen[i] = candidate_offsets[np.argmin(costs)]

    return chosen


def estimate_best_offset_for_scene(nusc, scene_token, candidate_offsets):
    """
    Evaluates alignment error for each candidate offset and selects
    the best frame-wise estimate.

    compute_scene_radar_spread_timeseries() is the main alignment
    metric function from the full implementation.
    """
    raw_errors = {}

    for offset in candidate_offsets:
        df = compute_scene_radar_spread_timeseries(
            nusc,
            scene_token,
            pose_time_offset_ms = offset,
            window_s = 2.0,
            max_sweeps = 40
        )

        raw_errors[offset] = df["alignment_error_xy"].to_numpy()

    errors = np.stack([raw_errors[o] for o in candidate_offsets], axis = 1)
    best_idx = np.nanargmin(errors, axis = 1)

    return np.array(candidate_offsets)[best_idx], raw_errors


def recovery_pct(clean, drifted, corrected):
    """
    Percentage recovery toward clean alignment.
    """
    return 100 * (drifted - corrected) / (drifted - clean)


# --------------------------------------------------
# Example experiment (Section 5.2.2)
# --------------------------------------------------

scene_token = nusc.scene[0]["token"]
candidate_offsets = tuple(range(-300, 301, 50))

# Simulated drift corruption
drift_fn = drift_offset_fn(a_ms_per_frame = 0.5, jitter_ms = 5.0)

df_clean = compute_scene_radar_spread_timeseries(
    nusc, scene_token, pose_time_offset_ms = 0.0
)

df_drifted = compute_scene_radar_spread_timeseries(
    nusc, scene_token, pose_time_offset_ms = drift_fn
)

# Raw frame-wise offset estimates
best_offsets, raw_errors = estimate_best_offset_for_scene(
    nusc,
    scene_token,
    candidate_offsets
)

# Smoothed correction
smoothed_offsets = smooth_offsets(best_offsets)

# Continuity-aware correction
continuity_offsets = estimate_offsets_with_continuity(
    raw_errors,
    candidate_offsets,
    lam = 0.02
)

continuity_offsets = smooth_offsets(continuity_offsets)

df_corrected = compute_scene_radar_spread_timeseries(
    nusc,
    scene_token,
    pose_time_offset_ms = lambda i: continuity_offsets[min(i, len(continuity_offsets) - 1)]
)

print("Clean mean error:", summarise_alignment(df_clean))
print("Drifted mean error:", summarise_alignment(df_drifted))
print("Corrected mean error:", summarise_alignment(df_corrected))


# Figure used in Results Section 5.2.2
plt.figure()
plt.plot(raw_errors[50], label = "corrupted")
plt.plot(df_corrected["alignment_error_xy"], label = "corrected")
plt.xlabel("LiDAR frame index")
plt.ylabel("Alignment error (m)")
plt.title("Temporal alignment error before and after correction")
plt.legend()
plt.show()

# 5.3 Spatial and Temporal Degradation

In [ ]:
# Appendix Code 3: Spatial and Temporal Degradation Experiments


def get_scene_samples(nusc, scene_index = 0):
    """
    Returns all nuScenes samples in a selected scene in temporal order.
    """
    scene = nusc.scene[scene_index]
    sample_token = scene["first_sample_token"]

    samples = []

    while sample_token:
        sample = nusc.get("sample", sample_token)
        samples.append(sample)
        sample_token = sample["next"]

    return samples


def fuse_frame_with_spatial_offset(
    nusc,
    sample,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    dx = 0.0,
    dy = 0.0,
    dz = 0.0,
):
    """
    Applies an artificial translation to radar points before LiDAR-radar
    association. This simulates extrinsic calibration error.
    """
    lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])

    radar_pc, _ = RadarPointCloud.from_file_multisweep(
        nusc,
        sample,
        chan = radar_chan,
        ref_chan = ref_chan,
        nsweeps = 1
    )

    radar_xyz = radar_pc.points[:3, :].copy()
    radar_xyz[0, :] += dx
    radar_xyz[1, :] += dy
    radar_xyz[2, :] += dz

    radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    fused_objects = []

    for box_index, box in enumerate(boxes):
        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        velocity_xy = np.median(radar_vel_xy[:, inside_box], axis = 1)

        fused_objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": box.center[:2].copy(),
            "velocity_xy": velocity_xy.copy(),
            "radar_hits": hit_count
        })

    return fused_objects


def run_scene_with_spatial_offset(
    nusc,
    scene_index = 0,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    dx = 0.0,
    dy = 0.0,
    dz = 0.0,
):
    """
    Applies spatially degraded fusion to every frame in a scene.
    """
    samples = get_scene_samples(nusc, scene_index)
    all_frames = []

    for frame_index, sample in enumerate(samples):
        fused_objects = fuse_frame_with_spatial_offset(
            nusc,
            sample,
            radar_chan = radar_chan,
            ref_chan = ref_chan,
            min_hits = min_hits,
            dx = dx,
            dy = dy,
            dz = dz,
        )

        all_frames.append({
            "frame_index": frame_index,
            "sample_token": sample["token"],
            "objects": fused_objects
        })

    return all_frames


def fuse_frame_with_temporal_offset(
    nusc,
    lidar_sample,
    radar_sample,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
):
    """
    Uses LiDAR boxes from one frame and radar points from another frame.
    This simulates frame-level temporal misalignment.
    """
    lidar_sd = nusc.get("sample_data", lidar_sample["data"][ref_chan])

    radar_pc, _ = RadarPointCloud.from_file_multisweep(
        nusc,
        radar_sample,
        radar_chan,
        ref_chan,
        nsweeps = 1
    )

    radar_xyz = radar_pc.points[:3, :]
    radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    fused_objects = []

    for box_index, box in enumerate(boxes):
        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        velocity_xy = np.median(radar_vel_xy[:, inside_box], axis = 1)

        fused_objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": box.center[:2].copy(),
            "velocity_xy": velocity_xy.copy(),
            "radar_hits": hit_count
        })

    return fused_objects


def run_scene_with_temporal_offset(
    nusc,
    scene_index = 0,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    frame_offset = 0,
):
    """
    Applies temporal degradation by pairing each LiDAR frame with a shifted
    radar frame.

    frame_offset = 0   aligned baseline
    frame_offset = -1  previous radar frame
    frame_offset = +1  next radar frame
    """
    samples = get_scene_samples(nusc, scene_index)
    all_frames = []

    for frame_index, lidar_sample in enumerate(samples):
        radar_index = frame_index + frame_offset

        if radar_index < 0 or radar_index >= len(samples):
            continue

        radar_sample = samples[radar_index]

        fused_objects = fuse_frame_with_temporal_offset(
            nusc,
            lidar_sample = lidar_sample,
            radar_sample = radar_sample,
            radar_chan = radar_chan,
            ref_chan = ref_chan,
            min_hits = min_hits,
        )

        all_frames.append({
            "frame_index": frame_index,
            "radar_frame_index": radar_index,
            "sample_token": lidar_sample["token"],
            "objects": fused_objects
        })

    return all_frames



# Spatial degradation experiment used in Section 5.3.1
dx_offsets = [0.0, 0.2, 0.5, 1.0, 2.0]
spatial_results = []

for dx in dx_offsets:
    frames = run_scene_with_spatial_offset(
        nusc,
        scene_index = 0,
        dx = dx,
        dy = 0.0,
        dz = 0.0
    )

    metrics = summarise_fusion_results(frames)  # defined in Appendix A.1
    metrics["dx"] = dx
    spatial_results.append(metrics)


# Temporal degradation experiment used in Section 5.3.2
frame_offsets = [0, -1, 1, -2, 2]
temporal_results = []

for frame_offset in frame_offsets:
    frames = run_scene_with_temporal_offset(
        nusc,
        scene_index = 0,
        frame_offset = frame_offset
    )

    metrics = summarise_fusion_results(frames)  # defined in Appendix A.1
    metrics["frame_offset"] = frame_offset
    temporal_results.append(metrics)

# Yaw degradation experiment used for the yaw table in Section 5.3.1

def rotate_radar_points_yaw(radar_xyz, yaw_deg):
    """
    Applies yaw rotation to radar points in the LiDAR frame.
    """
    theta = np.deg2rad(yaw_deg)

    rotation = np.array([
        [np.cos(theta), -np.sin(theta), 0.0],
        [np.sin(theta),  np.cos(theta), 0.0],
        [0.0,            0.0,           1.0]
    ])

    return rotation @ radar_xyz


def fuse_frame_with_yaw_offset(
    nusc,
    sample,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    yaw_deg = 0.0,
):
    """
    Applies artificial yaw misalignment to radar points before association.
    """
    lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])

    radar_pc, _ = RadarPointCloud.from_file_multisweep(
        nusc,
        sample,
        chan = radar_chan,
        ref_chan = ref_chan,
        nsweeps = 1
    )

    radar_xyz = rotate_radar_points_yaw(radar_pc.points[:3, :].copy(), yaw_deg)
    radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))

    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    objects = []

    for box_index, box in enumerate(boxes):
        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        points_xy = radar_xyz[:2, inside_box].T
        velocity_xy = radar_vel_xy[:, inside_box].T
        centre_xy = box.center[:2].copy()

        v_est = compute_weighted_velocity(points_xy, velocity_xy, centre_xy)

        objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": centre_xy,
            "velocity_xy": v_est,
            "radar_hits": hit_count
        })

    return objects


def run_scene_with_yaw_offset(nusc, scene_index = 0, yaw_deg = 0.0):
    """
    Runs the yaw degradation experiment across a full scene.
    """
    samples = get_scene_samples(nusc, scene_index)
    frames = []

    for frame_index, sample in enumerate(samples):
        objects = fuse_frame_with_yaw_offset(
            nusc,
            sample,
            yaw_deg = yaw_deg
        )

        frames.append({
            "frame_index": frame_index,
            "sample_token": sample["token"],
            "objects": objects
        })

    return frames


yaw_offsets = [0.0, 1.0, 3.0, 5.0, 10.0]
yaw_results = []

for yaw in yaw_offsets:
    frames = run_scene_with_yaw_offset(nusc, scene_index = 0, yaw_deg = yaw)
    metrics = summarise_fusion_results(frames)
    metrics["yaw_deg"] = yaw
    yaw_results.append(metrics)
    

# Example plotting code for the main degradation figures
plt.figure()
plt.plot(
    [r["dx"] for r in spatial_results],
    [r["total_fused_objects"] for r in spatial_results],
    marker = "o"
)
plt.xlabel("Radar x-offset dx (m)")
plt.ylabel("Total fused objects")
plt.title("Effect of spatial misalignment on fusion count")
plt.grid(True)
plt.show()


plt.figure()
plt.bar(
    [str(r["frame_offset"]) for r in temporal_results],
    [r["total_fused_objects"] for r in temporal_results]
)
plt.xlabel("Frame offset")
plt.ylabel("Total fused objects")
plt.title("Effect of frame-level temporal offset")
plt.show()

# 5.4 Alignment Recovery and Estimation

In [ ]:
# Appendix Code 4: Spatial Alignment Recovery and Physically Constrained Scoring


def apply_spatial_transform(radar_xyz, dx = 0.0, dy = 0.0, dz = 0.0, yaw_deg = 0.0):
    """
    Applies translation and yaw rotation to radar points already expressed
    in the LiDAR coordinate frame.
    """
    pts = radar_xyz.copy()

    theta = np.deg2rad(yaw_deg)
    rotation = np.array([
        [np.cos(theta), -np.sin(theta), 0.0],
        [np.sin(theta),  np.cos(theta), 0.0],
        [0.0,            0.0,           1.0]
    ])

    pts = rotation @ pts
    pts[0, :] += dx
    pts[1, :] += dy
    pts[2, :] += dz

    return pts


def score_scene_alignment(
    nusc,
    scene_index = 0,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    degrade_dx = 0.0,
    degrade_yaw_deg = 0.0,
    correct_dx = 0.0,
    correct_yaw_deg = 0.0,
    max_range = 60.0,
):
    """
    Scores a candidate spatial correction over a complete scene.

    The radar points are first degraded using a known artificial offset,
    then corrected using the candidate transform. The score is based on:
      - number of LiDAR boxes containing radar points
      - number of associated radar hits
      - radar-to-box-centre distance
    """
    scene = nusc.scene[scene_index]
    sample_token = scene["first_sample_token"]

    matched_boxes = 0
    total_hits = 0
    centre_distances = []
    frames_used = 0

    while sample_token:
        sample = nusc.get("sample", sample_token)
        lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])

        radar_pc, _ = RadarPointCloud.from_file_multisweep(
            nusc,
            sample,
            chan = radar_chan,
            ref_chan = ref_chan,
            nsweeps = 1
        )

        radar_xyz = radar_pc.points[:3, :].copy()

        # Apply artificial degradation, then candidate correction.
        radar_xyz = apply_spatial_transform(
            radar_xyz,
            dx = degrade_dx,
            yaw_deg = degrade_yaw_deg
        )

        radar_xyz = apply_spatial_transform(
            radar_xyz,
            dx = correct_dx,
            yaw_deg = correct_yaw_deg
        )

        _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

        for box in boxes:
            if np.linalg.norm(box.center[:2]) > max_range:
                continue

            inside_box = points_in_box(box, radar_xyz)
            hit_count = int(np.sum(inside_box))

            total_hits += hit_count

            if hit_count > 0:
                matched_boxes += 1

                # Physical constraint: radar points should be close to
                # the centre of the LiDAR box they are associated with.
                radar_points_xy = radar_xyz[:2, inside_box].T
                distances = np.linalg.norm(
                    radar_points_xy - box.center[:2],
                    axis = 1
                )
                centre_distances.extend(distances.tolist())

        frames_used += 1
        sample_token = sample["next"]

    mean_distance = (
        float(np.mean(centre_distances))
        if centre_distances else np.inf
    )

    return {
        "frames_used": frames_used,
        "matched_boxes": matched_boxes,
        "total_hits": total_hits,
        "mean_centre_distance": mean_distance
    }


def estimate_spatial_correction(
    nusc,
    scene_index = 0,
    degrade_dx = 0.0,
    degrade_yaw_deg = 0.0,
    dx_candidates = (0.0,),
    yaw_candidates = (0.0,),
    w_boxes = 1000.0,
    w_hits = 10.0,
    w_distance = 50.0,
):
    """
    Grid-searches candidate spatial corrections using a physically
    constrained score:

        score = w_boxes * matched_boxes
              + w_hits * total_hits
              - w_distance * mean_centre_distance

    Larger scores indicate better alignment.
    """
    best_result = None
    best_score = -np.inf
    all_results = []

    for dx in dx_candidates:
        for yaw in yaw_candidates:
            stats = score_scene_alignment(
                nusc,
                scene_index = scene_index,
                degrade_dx = degrade_dx,
                degrade_yaw_deg = degrade_yaw_deg,
                correct_dx = dx,
                correct_yaw_deg = yaw
            )

            score = (
                w_boxes * stats["matched_boxes"]
                + w_hits * stats["total_hits"]
                - w_distance * stats["mean_centre_distance"]
            )

            result = {
                "candidate_dx": dx,
                "candidate_yaw_deg": yaw,
                "matched_boxes": stats["matched_boxes"],
                "total_hits": stats["total_hits"],
                "mean_centre_distance": stats["mean_centre_distance"],
                "score": score
            }

            all_results.append(result)

            if score > best_score:
                best_score = score
                best_result = result

    return best_result, all_results


# Translation recovery experiment used in Section 5.4.1
best_translation, translation_candidates = estimate_spatial_correction(
    nusc,
    scene_index = 0,
    degrade_dx = 1.0,
    degrade_yaw_deg = 0.0,
    dx_candidates = np.linspace(-1.5, 0.5, 9),
    yaw_candidates = (0.0,)
)

# Yaw recovery experiment used in Section 5.4.2
best_yaw, yaw_candidates = estimate_spatial_correction(
    nusc,
    scene_index = 0,
    degrade_dx = 0.0,
    degrade_yaw_deg = 3.0,
    dx_candidates = (0.0,),
    yaw_candidates = (-5.0, -4.0, -3.0, -2.0, -1.0, 0.0)
)

print("Best translation correction:", best_translation)
print("Best yaw correction:", best_yaw)

Best translation correction: {'candidate_dx': np.float64(-0.75), 'candidate_yaw_deg': 0.0, 'matched_boxes': 178, 'total_hits': 295, 'mean_centre_distance': 1.5125460022108814, 'score': 180874.37269988944}
Best yaw correction: {'candidate_dx': 0.0, 'candidate_yaw_deg': -3.0, 'matched_boxes': 176, 'total_hits': 291, 'mean_centre_distance': 1.5149704526228616, 'score': 178834.25147736885}


# 5.5 Temporal Recovery and Tracking

In [ ]:
# Appendix Code 5: Temporal Recovery, Kalman Tracking, and Multiframe Fusion


# --------------------------------------------------
# A.5.1 Shared helper functions
# --------------------------------------------------

def compute_weighted_velocity(points_xy, radar_vel_xy, box_center_xy, eps = 1e-6):
    """
    Estimates object velocity from associated radar points.

    Radar points closer to the LiDAR box centre are given larger weights.
    This reduces the influence of points near the boundary of the object box.
    """
    dists = np.linalg.norm(points_xy - box_center_xy[None, :], axis = 1)
    weights = 1.0 / (dists + eps)

    if np.sum(weights) <= 0:
        return np.mean(radar_vel_xy, axis = 0)

    return np.sum(radar_vel_xy * weights[:, None], axis = 0) / np.sum(weights)


def summarise_results(frames):
    """
    Computes object-level summary metrics used in Section 5.5.
    """
    hits = []
    speeds = []

    for frame in frames:
        for obj in frame["objects"]:
            hits.append(obj["radar_hits"])
            vx, vy = obj["velocity_xy"]
            speeds.append(np.sqrt(vx**2 + vy**2))

    total_frames = len(frames)
    total_objects = sum(len(frame["objects"]) for frame in frames)

    return {
        "total_frames": total_frames,
        "total_fused_objects": total_objects,
        "objects_per_frame": total_objects / total_frames if total_frames else 0.0,
        "avg_hits": float(np.mean(hits)) if hits else 0.0,
        "median_hits": float(np.median(hits)) if hits else 0.0,
        "avg_speed": float(np.mean(speeds)) if speeds else 0.0,
        "median_speed": float(np.median(speeds)) if speeds else 0.0,
    }


# --------------------------------------------------
# A.5.2 Baseline and temporally degraded measurements
# --------------------------------------------------

def extract_temporally_offset_measurements(nusc, lidar_sample, radar_sample,
                                           radar_chan = "RADAR_FRONT",
                                           ref_chan = "LIDAR_TOP",
                                           min_hits = 1,
                                           max_range = 60.0):
    """
    Uses LiDAR boxes from lidar_sample and radar points from radar_sample.
    This simulates temporal misalignment.
    """
    lidar_sd = nusc.get("sample_data", lidar_sample["data"][ref_chan])

    radar_pc, _ = RadarPointCloud.from_file_multisweep(
        nusc,
        radar_sample,
        chan = radar_chan,
        ref_chan = ref_chan,
        nsweeps = 1
    )

    radar_xyz = radar_pc.points[:3, :]
    radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    objects = []

    for box_index, box in enumerate(boxes):
        if np.linalg.norm(box.center[:2]) > max_range:
            continue

        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        points_xy = radar_xyz[:2, inside_box].T
        velocity_xy = radar_vel_xy[:, inside_box].T
        centre_xy = box.center[:2].copy()

        v_est = compute_weighted_velocity(points_xy, velocity_xy, centre_xy)

        objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": centre_xy,
            "velocity_xy": v_est,
            "radar_hits": hit_count,
            "associated_radar_points_xy": points_xy.tolist(),
        })

    return objects


def run_scene_measurements(nusc, scene_index = 0, frame_offset = 0):
    """
    Runs object-level fusion across a scene.

    frame_offset = 0 gives the aligned baseline.
    frame_offset = -1 or +1 simulates temporal misalignment by using radar
    measurements from a neighbouring frame.
    """
    samples = get_scene_samples(nusc, scene_index)
    frames = []

    for i, lidar_sample in enumerate(samples):
        radar_index = i + frame_offset

        if radar_index < 0 or radar_index >= len(samples):
            continue

        radar_sample = samples[radar_index]

        # For baseline, lidar_sample and radar_sample are the same.
        # For degraded cases, LiDAR boxes and radar points come from different frames.
        objects = extract_temporally_offset_measurements(
            nusc,
            lidar_sample = lidar_sample,
            radar_sample = radar_sample
        )

        frames.append({
            "frame_index": i,
            "timestamp": lidar_sample["timestamp"],
            "objects": objects
        })

    return frames


# --------------------------------------------------
# A.5.3 Interpolation-based temporal recovery
# --------------------------------------------------

def interpolate_velocity(v_prev, v_next, alpha):
    """
    Linearly interpolates velocity between previous and next radar-derived estimates.
    """
    return (1.0 - alpha) * v_prev + alpha * v_next


def extract_interpolated_measurements(
    nusc,
    prev_sample,
    curr_sample,
    next_sample,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    max_range = 60.0,
):
    """
    Uses LiDAR boxes from the current frame, but associates radar points from
    previous and next frames. Velocity is then interpolated to the current time.
    """
    lidar_sd = nusc.get("sample_data", curr_sample["data"][ref_chan])
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    def load_radar(sample):
        radar_pc, _ = RadarPointCloud.from_file_multisweep(
            nusc,
            sample,
            chan = radar_chan,
            ref_chan = ref_chan,
            nsweeps = 1
        )
        radar_xyz = radar_pc.points[:3, :]
        radar_vel_xy = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))
        return radar_xyz, radar_vel_xy

    radar_prev, vel_prev = load_radar(prev_sample)
    radar_next, vel_next = load_radar(next_sample)

    t_prev = prev_sample["timestamp"]
    t_curr = curr_sample["timestamp"]
    t_next = next_sample["timestamp"]

    alpha = 0.5 if t_next == t_prev else (t_curr - t_prev) / (t_next - t_prev)

    objects = []

    for box_index, box in enumerate(boxes):
        if np.linalg.norm(box.center[:2]) > max_range:
            continue

        mask_prev = points_in_box(box, radar_prev)
        mask_next = points_in_box(box, radar_next)

        hit_prev = int(np.sum(mask_prev))
        hit_next = int(np.sum(mask_next))

        if hit_prev < min_hits or hit_next < min_hits:
            continue

        centre_xy = box.center[:2].copy()

        v_prev = compute_weighted_velocity(
            radar_prev[:2, mask_prev].T,
            vel_prev[:, mask_prev].T,
            centre_xy
        )

        v_next = compute_weighted_velocity(
            radar_next[:2, mask_next].T,
            vel_next[:, mask_next].T,
            centre_xy
        )

        v_interp = interpolate_velocity(v_prev, v_next, alpha)

        associated_points = np.vstack([
            radar_prev[:2, mask_prev].T,
            radar_next[:2, mask_next].T
        ])

        objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": centre_xy,
            "velocity_xy": v_interp,
            "radar_hits": hit_prev + hit_next,
            "associated_radar_points_xy": associated_points.tolist(),
        })

    return objects


def run_scene_interpolation(nusc, scene_index = 0):
    """
    Applies interpolation recovery across a scene.
    First and last frames are skipped because interpolation requires neighbours.
    """
    samples = get_scene_samples(nusc, scene_index)
    frames = []

    for i in range(1, len(samples) - 1):
        objects = extract_interpolated_measurements(
            nusc,
            prev_sample = samples[i - 1],
            curr_sample = samples[i],
            next_sample = samples[i + 1],
        )

        frames.append({
            "frame_index": i,
            "timestamp": samples[i]["timestamp"],
            "objects": objects
        })

    return frames


def evaluate_interpolation_recovery(nusc, scene_index = 0):
    """
    Compares baseline, temporally degraded, and interpolated measurements.
    This produced the interpolation table in Section 5.5.1.
    """
    baseline = run_scene_measurements(nusc, scene_index, frame_offset = 0)
    degraded_prev = run_scene_measurements(nusc, scene_index, frame_offset = -1)
    degraded_next = run_scene_measurements(nusc, scene_index, frame_offset = 1)
    interpolated = run_scene_interpolation(nusc, scene_index)

    return {
        "baseline": summarise_results(baseline),
        "degraded_prev": summarise_results(degraded_prev),
        "degraded_next": summarise_results(degraded_next),
        "interpolated": summarise_results(interpolated),
    }


# --------------------------------------------------
# A.5.4 Constant-velocity Kalman tracking
# --------------------------------------------------

class KalmanTrack:
    """
    Constant-velocity Kalman track with state:
        x = [px, py, vx, vy]
    """
    def __init__(self, track_id, initial_state, timestamp, class_name = None):
        self.track_id = track_id
        self.x = initial_state.copy()
        self.P = np.diag([1.0, 1.0, 2.0, 2.0])
        self.last_timestamp = timestamp
        self.class_name = class_name
        self.missed = 0
        self.history = [initial_state.copy()]

    def predict(self, dt, q_pos = 1.0, q_vel = 1.0):
        F = np.array([
            [1.0, 0.0, dt,  0.0],
            [0.0, 1.0, 0.0, dt ],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0]
        ])

        Q = np.diag([
            q_pos * dt**2,
            q_pos * dt**2,
            q_vel * dt,
            q_vel * dt
        ])

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + Q

    def update(self, measurement, R):
        H = np.eye(4)

        residual = measurement - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ residual
        self.P = (np.eye(4) - K @ H) @ self.P

        self.missed = 0
        self.history.append(self.x.copy())

    def mark_missed(self):
        self.missed += 1
        self.history.append(self.x.copy())


def measurement_from_object(obj):
    """
    Converts a fused object into a Kalman measurement:
        z = [px, py, vx, vy]
    """
    px, py = obj["position_xy"]
    vx, vy = obj["velocity_xy"]
    return np.array([px, py, vx, vy], dtype = float)


def measurement_covariance(obj, pos_var = 1.0, vel_var_base = 2.0):
    """
    Uses radar hit count to scale velocity uncertainty.
    More radar points imply slightly lower velocity uncertainty.
    """
    hits = max(obj["radar_hits"], 1)
    vel_var = vel_var_base / hits
    return np.diag([pos_var, pos_var, vel_var, vel_var])


def associate_tracks_to_measurements(tracks, measurements, gate_dist = 3.0):
    """
    Greedy nearest-neighbour association using position distance.
    """
    if len(tracks) == 0 or len(measurements) == 0:
        return [], list(range(len(tracks))), list(range(len(measurements)))

    track_pos = np.array([track.x[:2] for track in tracks])
    meas_pos = np.array([meas[:2] for meas in measurements])

    distance_matrix = np.linalg.norm(
        track_pos[:, None, :] - meas_pos[None, :, :],
        axis = 2
    )

    matches = []
    used_tracks = set()
    used_measurements = set()

    while True:
        best_pair = None
        best_distance = np.inf

        for track_index in range(len(tracks)):
            if track_index in used_tracks:
                continue

            for meas_index in range(len(measurements)):
                if meas_index in used_measurements:
                    continue

                distance = distance_matrix[track_index, meas_index]

                if distance < best_distance:
                    best_distance = distance
                    best_pair = (track_index, meas_index)

        if best_pair is None or best_distance > gate_dist:
            break

        track_index, meas_index = best_pair
        matches.append((track_index, meas_index))
        used_tracks.add(track_index)
        used_measurements.add(meas_index)

    unmatched_tracks = [
        i for i in range(len(tracks))
        if i not in used_tracks
    ]

    unmatched_measurements = [
        i for i in range(len(measurements))
        if i not in used_measurements
    ]

    return matches, unmatched_tracks, unmatched_measurements


def run_kalman_tracking_on_frames(frames, gate_dist = 3.0, max_missed = 2):
    """
    Runs constant-velocity Kalman tracking on a sequence of fused frame measurements.
    """
    active_tracks = []
    finished_tracks = []
    next_track_id = 0
    previous_timestamp = None

    for frame in frames:
        timestamp = frame["timestamp"]
        objects = frame["objects"]

        dt = 0.1 if previous_timestamp is None else (timestamp - previous_timestamp) / 1e6

        for track in active_tracks:
            track.predict(dt)

        measurements = [measurement_from_object(obj) for obj in objects]
        covariances = [measurement_covariance(obj) for obj in objects]

        matches, unmatched_tracks, unmatched_measurements = associate_tracks_to_measurements(
            active_tracks,
            measurements,
            gate_dist = gate_dist
        )

        for track_index, meas_index in matches:
            active_tracks[track_index].update(
                measurements[meas_index],
                covariances[meas_index]
            )

        for track_index in unmatched_tracks:
            active_tracks[track_index].mark_missed()

        for meas_index in unmatched_measurements:
            new_track = KalmanTrack(
                track_id = next_track_id,
                initial_state = measurements[meas_index],
                timestamp = timestamp,
                class_name = objects[meas_index]["box_name"]
            )
            active_tracks.append(new_track)
            next_track_id += 1

        still_active = []

        for track in active_tracks:
            if track.missed > max_missed:
                finished_tracks.append(track)
            else:
                still_active.append(track)

        active_tracks = still_active
        previous_timestamp = timestamp

    finished_tracks.extend(active_tracks)
    return finished_tracks


def summarise_tracks(tracks):
    """
    Computes track-level metrics used in Sections 5.5.2 and 5.5.4.
    """
    lengths = [len(track.history) for track in tracks]
    speeds = [np.linalg.norm(track.x[2:4]) for track in tracks]

    return {
        "num_tracks": len(tracks),
        "avg_track_length": float(np.mean(lengths)) if lengths else 0.0,
        "median_track_length": float(np.median(lengths)) if lengths else 0.0,
        "avg_final_speed": float(np.mean(speeds)) if speeds else 0.0,
        "median_final_speed": float(np.median(speeds)) if speeds else 0.0,
    }


def evaluate_kalman_temporal_cases(nusc, scene_index = 0):
    """
    Evaluates Kalman tracking under baseline, degraded, and interpolation conditions.
    This corresponds to Section 5.5.2.
    """
    baseline_frames = run_scene_measurements(nusc, scene_index, frame_offset = 0)
    degraded_frames = run_scene_measurements(nusc, scene_index, frame_offset = -1)
    interpolated_frames = run_scene_interpolation(nusc, scene_index)

    baseline_tracks = run_kalman_tracking_on_frames(baseline_frames)
    degraded_tracks = run_kalman_tracking_on_frames(degraded_frames)
    interpolated_tracks = run_kalman_tracking_on_frames(interpolated_frames)

    return {
        "baseline": summarise_tracks(baseline_tracks),
        "degraded": summarise_tracks(degraded_tracks),
        "interpolated": summarise_tracks(interpolated_tracks),
    }


# --------------------------------------------------
# A.5.5 Multiframe fusion
# --------------------------------------------------

def get_adjacent_samples(nusc, sample):
    """
    Returns previous and next samples for a nuScenes sample.
    """
    prev_sample = nusc.get("sample", sample["prev"]) if sample["prev"] else None
    next_sample = nusc.get("sample", sample["next"]) if sample["next"] else None
    return prev_sample, next_sample


def extract_multiframe_measurements(
    nusc,
    sample,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    max_range = 60.0,
):
    """
    Aggregates radar points from previous, current, and next frames before
    associating them with current LiDAR boxes.
    """
    lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    prev_sample, next_sample = get_adjacent_samples(nusc, sample)
    samples_to_use = [s for s in [prev_sample, sample, next_sample] if s is not None]

    all_points = []
    all_velocities = []

    for radar_sample in samples_to_use:
        radar_pc, _ = RadarPointCloud.from_file_multisweep(
            nusc,
            radar_sample,
            chan = radar_chan,
            ref_chan = ref_chan,
            nsweeps = 1
        )

        all_points.append(radar_pc.points[:3, :])
        all_velocities.append(np.vstack((radar_pc.points[8, :], radar_pc.points[9, :])))

    radar_xyz = np.hstack(all_points)
    radar_vel_xy = np.hstack(all_velocities)

    objects = []

    for box_index, box in enumerate(boxes):
        if np.linalg.norm(box.center[:2]) > max_range:
            continue

        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        points_xy = radar_xyz[:2, inside_box].T
        velocity_xy = radar_vel_xy[:, inside_box].T
        centre_xy = box.center[:2].copy()

        v_est = compute_weighted_velocity(points_xy, velocity_xy, centre_xy)

        objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": centre_xy,
            "velocity_xy": v_est,
            "radar_hits": hit_count,
            "associated_radar_points_xy": points_xy.tolist(),
        })

    return objects


def run_scene_multiframe_fusion(nusc, scene_index = 0):
    """
    Runs naive multiframe fusion across a scene.
    """
    samples = get_scene_samples(nusc, scene_index)
    frames = []

    for frame_index, sample in enumerate(samples):
        objects = extract_multiframe_measurements(nusc, sample)

        frames.append({
            "frame_index": frame_index,
            "timestamp": sample["timestamp"],
            "objects": objects
        })

    return frames


def evaluate_multiframe_fusion(nusc, scene_index = 0):
    """
    Compares baseline, degraded, and multiframe fusion.
    This corresponds to Section 5.5.3.
    """
    baseline = run_scene_measurements(nusc, scene_index, frame_offset = 0)
    degraded = run_scene_measurements(nusc, scene_index, frame_offset = -1)
    multiframe = run_scene_multiframe_fusion(nusc, scene_index)

    return {
        "baseline": summarise_results(baseline),
        "degraded": summarise_results(degraded),
        "multiframe": summarise_results(multiframe),
    }


# --------------------------------------------------
# A.5.6 Temporal-weighted multiframe fusion
# --------------------------------------------------

def compute_temporal_weighted_velocity(
    points_xy,
    radar_vel_xy,
    box_center_xy,
    frame_offsets,
    alpha = 0.7,
    eps = 1e-6
):
    """
    Combines spatial and temporal weighting.

    Spatial weighting favours radar points close to the LiDAR box centre.
    Temporal weighting favours radar points from the current frame.
    """
    spatial_dist = np.linalg.norm(points_xy - box_center_xy[None, :], axis = 1)
    spatial_weights = 1.0 / (spatial_dist + eps)

    temporal_weights = np.exp(-alpha * np.abs(frame_offsets))

    weights = spatial_weights * temporal_weights

    if np.sum(weights) <= 0:
        return np.mean(radar_vel_xy, axis = 0)

    return np.sum(radar_vel_xy * weights[:, None], axis = 0) / np.sum(weights)


def extract_temporal_weighted_multiframe_measurements(
    nusc,
    samples,
    frame_index,
    radar_chan = "RADAR_FRONT",
    ref_chan = "LIDAR_TOP",
    min_hits = 1,
    max_range = 60.0,
):
    """
    Weighted multiframe fusion using radar from frames t-1, t, and t+1.
    """
    sample = samples[frame_index]
    lidar_sd = nusc.get("sample_data", sample["data"][ref_chan])
    _, boxes, _ = nusc.get_sample_data(lidar_sd["token"])

    radar_points = []
    radar_velocities = []
    frame_offsets = []

    for offset in [-1, 0, 1]:
        neighbour_index = frame_index + offset

        if neighbour_index < 0 or neighbour_index >= len(samples):
            continue

        radar_sample = samples[neighbour_index]

        radar_pc, _ = RadarPointCloud.from_file_multisweep(
            nusc,
            radar_sample,
            chan = radar_chan,
            ref_chan = ref_chan,
            nsweeps = 1
        )

        pts = radar_pc.points[:3, :]
        vel = np.vstack((radar_pc.points[8, :], radar_pc.points[9, :]))

        radar_points.append(pts)
        radar_velocities.append(vel)
        frame_offsets.extend([offset] * pts.shape[1])

    radar_xyz = np.hstack(radar_points)
    radar_vel_xy = np.hstack(radar_velocities)
    frame_offsets = np.array(frame_offsets)

    objects = []

    for box_index, box in enumerate(boxes):
        if np.linalg.norm(box.center[:2]) > max_range:
            continue

        inside_box = points_in_box(box, radar_xyz)
        hit_count = int(np.sum(inside_box))

        if hit_count < min_hits:
            continue

        points_xy = radar_xyz[:2, inside_box].T
        velocity_xy = radar_vel_xy[:, inside_box].T
        offsets = frame_offsets[inside_box]

        v_est = compute_temporal_weighted_velocity(
            points_xy,
            velocity_xy,
            box.center[:2],
            offsets
        )

        objects.append({
            "box_index": box_index,
            "box_name": box.name,
            "position_xy": box.center[:2].copy(),
            "velocity_xy": v_est,
            "radar_hits": hit_count,
            "associated_radar_points_xy": points_xy.tolist(),
        })

    return objects


def run_temporal_weighted_multiframe(nusc, scene_index = 0):
    """
    Runs temporal-weighted multiframe fusion across a scene.
    """
    samples = get_scene_samples(nusc, scene_index)
    frames = []

    for frame_index, sample in enumerate(samples):
        objects = extract_temporal_weighted_multiframe_measurements(
            nusc,
            samples,
            frame_index
        )

        frames.append({
            "frame_index": frame_index,
            "timestamp": sample["timestamp"],
            "objects": objects
        })

    return frames


# --------------------------------------------------
# A.5.7 Track-level comparison for multiframe fusion
# --------------------------------------------------

def evaluate_track_level_multiframe(nusc, scene_index = 0):
    """
    Runs the track-level comparison used in Section 5.5.4.
    """
    baseline_frames = run_scene_measurements(nusc, scene_index, frame_offset = 0)
    degraded_frames = run_scene_measurements(nusc, scene_index, frame_offset = -1)
    multiframe_frames = run_scene_multiframe_fusion(nusc, scene_index)
    weighted_frames = run_temporal_weighted_multiframe(nusc, scene_index)

    baseline_tracks = run_kalman_tracking_on_frames(baseline_frames)
    degraded_tracks = run_kalman_tracking_on_frames(degraded_frames)
    multiframe_tracks = run_kalman_tracking_on_frames(multiframe_frames)
    weighted_tracks = run_kalman_tracking_on_frames(weighted_frames)

    return {
        "baseline_measurements": summarise_results(baseline_frames),
        "degraded_measurements": summarise_results(degraded_frames),
        "multiframe_measurements": summarise_results(multiframe_frames),
        "weighted_measurements": summarise_results(weighted_frames),

        "baseline_tracks": summarise_tracks(baseline_tracks),
        "degraded_tracks": summarise_tracks(degraded_tracks),
        "multiframe_tracks": summarise_tracks(multiframe_tracks),
        "weighted_tracks": summarise_tracks(weighted_tracks),
    }


# --------------------------------------------------
# A.5.8 Constant-acceleration Kalman model
# --------------------------------------------------

class ConstantAccelerationKalmanTrack:
    """
    Constant-acceleration Kalman track.

    State:
        x = [px, py, vx, vy, ax, ay]

    Measurements:
        z = [px, py, vx, vy]
    """
    def __init__(self, track_id, initial_measurement, timestamp):
        px, py, vx, vy = initial_measurement

        self.track_id = track_id
        self.x = np.array([px, py, vx, vy, 0.0, 0.0], dtype = float)
        self.P = np.diag([1.0, 1.0, 2.0, 2.0, 5.0, 5.0])
        self.last_timestamp = timestamp
        self.missed = 0
        self.history = [self.x.copy()]

    def predict(self, dt, q = 1.0):
        F = np.array([
            [1.0, 0.0, dt,  0.0, 0.5 * dt**2, 0.0],
            [0.0, 1.0, 0.0, dt,  0.0,         0.5 * dt**2],
            [0.0, 0.0, 1.0, 0.0, dt,          0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0,         dt],
            [0.0, 0.0, 0.0, 0.0, 1.0,         0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0,         1.0]
        ])

        Q = np.eye(6) * q

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + Q

    def update(self, measurement):
        H = np.array([
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
        ])

        R = np.diag([1.0, 1.0, 2.0, 2.0])

        residual = measurement - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ residual
        self.P = (np.eye(6) - K @ H) @ self.P

        self.missed = 0
        self.history.append(self.x.copy())

    def mark_missed(self):
        self.missed += 1
        self.history.append(self.x.copy())


def summarise_constant_acceleration_tracks(tracks):
    """
    Summarises constant-acceleration tracks for comparison with the
    constant-velocity Kalman model.
    """
    lengths = [len(track.history) for track in tracks]
    speeds = [np.linalg.norm(track.x[2:4]) for track in tracks]

    return {
        "num_tracks": len(tracks),
        "avg_track_length": float(np.mean(lengths)) if lengths else 0.0,
        "avg_speed": float(np.mean(speeds)) if speeds else 0.0
    }
    
# --------------------------------------------------
# Example calls used to generate Section 5.5 result tables
# --------------------------------------------------

interpolation_results = evaluate_interpolation_recovery(nusc, scene_index = 0)
kalman_results = evaluate_kalman_temporal_cases(nusc, scene_index = 0)
multiframe_results = evaluate_multiframe_fusion(nusc, scene_index = 0)
track_level_results = evaluate_track_level_multiframe(nusc, scene_index = 0)

print(interpolation_results)
print(kalman_results)
print(multiframe_results)
print(track_level_results)